In [3]:
from pathlib import Path

print(Path.cwd())
print(list(Path.cwd().iterdir())[:10])

/Users/mell/predicting-predictability/data
[PosixPath('/Users/mell/predicting-predictability/data/drugseq_registry_overlap_status.csv'), PosixPath('/Users/mell/predicting-predictability/data/morgan_autoencoder_bottleneck_sweep_results.csv'), PosixPath('/Users/mell/predicting-predictability/data/perturbseqr_morgan_fingerprints.csv'), PosixPath('/Users/mell/predicting-predictability/data/umap_morgan_fingerprints.csv'), PosixPath('/Users/mell/predicting-predictability/data/july_wraps.ipynb'), PosixPath('/Users/mell/predicting-predictability/data/morgan_autoencoder_latent64_vectors.csv'), PosixPath('/Users/mell/predicting-predictability/data/drugseq_new_compounds_with_smiles.csv'), PosixPath('/Users/mell/predicting-predictability/data/pca_morgan_fingerprints.csv'), PosixPath('/Users/mell/predicting-predictability/data/.DS_Store'), PosixPath('/Users/mell/predicting-predictability/data/umap_maccs_keys.csv')]


In [4]:
from pathlib import Path
import pandas as pd
import json

folder = Path("Step 1 Files")

for f in sorted(folder.iterdir()):
    print(f.name)

.DS_Store
CREEDS.json
Drug_Repurposing_Hub.txt
LINCS_small_molecules.tsv
Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata.parquet
Novartis_metadata.tsv
Tahoe_100M.parquet
creeds-chem-drug_metadata.parquet
deepcover-moa-drug_metadata.parquet
lincs-l1000-cp-drug_metadata.parquet
microarrays-cmap-drug_metadata.parquet
pertseqr_metadata.parquet
rummageo-chem-drug_metadata.parquet
sciplex-drug_metadata.parquet
tahoe-100m-drug_metadata.parquet


In [5]:
from pathlib import Path
import pandas as pd
import json

folder = Path("Step 1 Files")

def load_file(path):
    suffix = path.suffix.lower()

    if path.name == ".DS_Store":
        return None

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix in [".tsv", ".txt"]:
        # most of these are tab-separated
        return pd.read_csv(path, sep="\t", low_memory=False)

    if suffix == ".json":
        with open(path, "r") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.json_normalize(data)
        elif isinstance(data, dict):
            return pd.json_normalize(data)
        else:
            return pd.DataFrame(data)

    return None


keywords = [
    "smiles", "canonical", "inchi", "inchikey",
    "pubchem", "cid", "compound", "drug",
    "pert", "name", "moa", "target"
]

inventory = []

for path in sorted(folder.iterdir()):
    if path.name == ".DS_Store":
        continue

    try:
        df = load_file(path)
        if df is None:
            continue

        matching_cols = [
            col for col in df.columns
            if any(k in str(col).lower() for k in keywords)
        ]

        inventory.append({
            "file": path.name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "matching_columns": matching_cols
        })

    except Exception as e:
        inventory.append({
            "file": path.name,
            "error": str(e)
        })

inventory_df = pd.DataFrame(inventory)
inventory_df

,file,rows,columns,matching_columns
0,CREEDS.json,875,15,"[smiles, pert_ids, drugbank_id, pubchem_cid, d..."
1,Drug_Repurposing_Hub.txt,22621,12,"[The Drug Repurposing Hub, Broad Institute, Un..."
2,LINCS_small_molecules.tsv,34418,8,"[Unnamed: 0, pert_name, target, moa, canonical..."
3,Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq...,4047,17,"[nibr_drug_name, pubchem_title, pubchem_cid, i..."
4,Novartis_metadata.tsv,4262,9,"[Unnamed: 0, inchi_key, smiles, moa, target, n..."
5,Tahoe_100M.parquet,379,9,"[drug, targets, moa-broad, moa-fine, canonical..."
6,creeds-chem-drug_metadata.parquet,270,12,"[creeds_drug_name, pubchem_title, pubchem_cid,..."
7,deepcover-moa-drug_metadata.parquet,874,12,"[deepcover_drug_name, pubchem_title, pubchem_c..."
8,lincs-l1000-cp-drug_metadata.parquet,5902,18,"[lincs_drug_name, pubchem_title, pubchem_cid, ..."
9,microarrays-cmap-drug_metadata.parquet,1309,12,"[cmap_drug_name, pubchem_title, pubchem_cid, d..."


In [6]:
for item in inventory:
    print("\n" + "="*80)
    print(item["file"])
    
    if "error" in item:
        print("ERROR:", item["error"])
    else:
        print("Rows:", item["rows"])
        print("Columns:", item["columns"])
        print("Matching columns:")
        for col in item["matching_columns"]:
            print("  -", col)


CREEDS.json
Rows: 875
Columns: 15
Matching columns:
  - smiles
  - pert_ids
  - drugbank_id
  - pubchem_cid
  - drug_name

Drug_Repurposing_Hub.txt
Rows: 22621
Columns: 12
Matching columns:
  - The Drug Repurposing Hub, Broad Institute
  - Unnamed: 2
  - Unnamed: 3
  - Unnamed: 4
  - Unnamed: 5
  - Unnamed: 6
  - Unnamed: 7
  - Unnamed: 8
  - Unnamed: 9
  - Unnamed: 10
  - Unnamed: 11

LINCS_small_molecules.tsv
Rows: 34418
Columns: 8
Matching columns:
  - Unnamed: 0
  - pert_name
  - target
  - moa
  - canonical_smiles
  - inchi_key
  - compound_aliases

Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata.parquet
Rows: 4047
Columns: 17
Matching columns:
  - nibr_drug_name
  - pubchem_title
  - pubchem_cid
  - inchi_key
  - smiles
  - moabox_moa
  - drh_moa
  - drh_target
  - drh_inchi_key
  - moa_inherited_from

Novartis_metadata.tsv
Rows: 4262
Columns: 9
Matching columns:
  - Unnamed: 0
  - inchi_key
  - smiles
  - moa
  - target
  - name
  - cid

Tahoe_100M.parquet
Rows: 37

In [7]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

inventory_df

,file,rows,columns,matching_columns
0,CREEDS.json,875,15,"[smiles, pert_ids, drugbank_id, pubchem_cid, drug_name]"
1,Drug_Repurposing_Hub.txt,22621,12,"[The Drug Repurposing Hub, Broad Institute, Unnamed: 2, Unnamed: 3, Unnamed: 4, Unnamed: 5, Unnamed: 6, Unnamed: 7, Unnamed: 8, Unnamed: 9, Unnamed: 10, Unnamed: 11]"
2,LINCS_small_molecules.tsv,34418,8,"[Unnamed: 0, pert_name, target, moa, canonical_smiles, inchi_key, compound_aliases]"
3,Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata.parquet,4047,17,"[nibr_drug_name, pubchem_title, pubchem_cid, inchi_key, smiles, moabox_moa, drh_moa, drh_target, drh_inchi_key, moa_inherited_from]"
4,Novartis_metadata.tsv,4262,9,"[Unnamed: 0, inchi_key, smiles, moa, target, name, cid]"
5,Tahoe_100M.parquet,379,9,"[drug, targets, moa-broad, moa-fine, canonical_smiles, pubchem_cid]"
6,creeds-chem-drug_metadata.parquet,270,12,"[creeds_drug_name, pubchem_title, pubchem_cid, drh_moa, drh_target, drh_inchi_key, moa_inherited_from]"
7,deepcover-moa-drug_metadata.parquet,874,12,"[deepcover_drug_name, pubchem_title, pubchem_cid, drh_moa, drh_target, drh_inchi_key, moa_inherited_from]"
8,lincs-l1000-cp-drug_metadata.parquet,5902,18,"[lincs_drug_name, pubchem_title, pubchem_cid, lincs_pert_id, inchi_key, smiles, lincs_target, lincs_moa, drh_moa, drh_target, drh_inchi_key, moa_inherited_from]"
9,microarrays-cmap-drug_metadata.parquet,1309,12,"[cmap_drug_name, pubchem_title, pubchem_cid, drh_moa, drh_target, drh_inchi_key, moa_inherited_from]"


In [8]:
inventory_df.to_csv("step1_file_inventory.csv", index=False)

In [9]:
for path in sorted(folder.iterdir()):
    if path.name == ".DS_Store":
        continue
    
    print("\n" + "="*100)
    print(path.name)
    
    df = load_file(path)
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())
    
    display(df.head(3))


CREEDS.json
Shape: (875, 15)
Columns:
['smiles', 'cell_type', 'pert_ids', 'platform', 'drugbank_id', 'curator', 'geo_id', 'pubchem_cid', 'drug_name', 'version', 'ctrl_ids', 'down_genes', 'up_genes', 'organism', 'id']


,smiles,cell_type,pert_ids,platform,drugbank_id,curator,geo_id,pubchem_cid,drug_name,version,ctrl_ids,down_genes,up_genes,organism,id
0,C1=C(C(=O)NC(=O)N1)F,Bone marrow Sca+ SP hematopoeitic stem cells (HSC) - 10 Day,"[GSM26744, GSM26745]",GPL81,DB00544,cadimo,GSE1559,3385.0,Fluorouracil,1.0,"[GSM26734, GSM26735]","[[Hmgn1, -0.17981423437595367], [Gm10260, -0.14166636765003204], [Npm1, -0.13669142127037048], [Fosb, -0.1250605285167694], [Gnb2l1, -0.12024638056755066], [Jund, -0.10381214320659637], [Ldha, -0.09215717762708664], [Eif4a2, -0.09049281477928162], [Igkv6-23, -0.08969537913799286], [Ccnd2, -0.08510024100542068], [Saraf, -0.08052777498960495], [Cd164, -0.07728368788957596], [AU020206, -0.07690908014774323], [Gm6793, -0.0701846107840538], [Ptma, -0.06753632426261902], [Serpina3g, -0.06303614377975464], [Ctnnb1, -0.06275594234466553], [Hspa9, -0.062107060104608536], [Ncl, -0.060621701180934906], [Gabarapl1, -0.05933355540037155], [Atp5a1, -0.05774950236082077], [Laptm4a, -0.05768873170018196], [Eef1a1, -0.05740978941321373], [Cd24a, -0.05661831051111221], [Rbbp7, -0.05167693272233009], [Pgk1, -0.051127053797245026], [Rpl6, -0.050143275409936905], [Wdr92, -0.049693748354911804], [Sptssa, -0.049358390271663666], [Hnrnpm, -0.04853145405650139], [Pum2, -0.0479293093085289], [Ppia, -0.04668370634317398], [Cd93, -0.04581252485513687], [Cct4, -0.044546984136104584], [Park7, -0.043298494070768356], [Gnai3, -0.042542047798633575], [H2-D1, -0.041550397872924805], [Pdha1, -0.04145074635744095], [Gtf2b, -0.04139845445752144], [Gm7367, -0.041331697255373], [Slc39a7, -0.041033245623111725], [Dhx9, -0.04079961031675339], [Tuba1b, -0.03999818116426468], [R3hdm1, -0.038771022111177444], [Rgs2, -0.03854523226618767], [Rpl8, -0.03717367723584175], [Tsc22d1, -0.03709498792886734], [Srsf2, -0.036761924624443054], [Plp2, -0.03673473373055458], [Socs2, -0.036646343767642975], [Bmi1, -0.036445051431655884], [Glrx3, -0.03629818186163902], [H2-K2, -0.035629305988550186], [Tbl1xr1, -0.03556510806083679], [Hspa5, -0.03555532172322273], [Arf1, -0.03532512113451958], [Atp5j, -0.035192567855119705], [Sumo3, -0.03517566993832588], [Zfand5, -0.03502270579338074], [Txn1, -0.03326527774333954], [Mcl1, -0.033099208027124405], [H3f3b, -0.03292763605713844], [Eef2, -0.03278762102127075], [Smarcc1, -0.0327836349606514], [Dusp1, -0.03197033330798149], [Sumo1, -0.031839825212955475], [Ubp1, -0.03138761967420578], [Cbfb, -0.031285449862480164], [Rbm8a, -0.031130339950323105], [Car2, -0.030882112681865692], [Ctdsp2, -0.03038480319082737], [Ptprs, -0.030013468116521835], [Wsb1, -0.029563892632722855], [Rac1, -0.029539447277784348], [Rpl28, -0.02941998466849327], [1110008F13Rik, -0.029407432302832603], [Nhp2, -0.029337916523218155], [Ski, -0.029260633513331413], [Lats2, -0.028955215588212013], [Rbm3, -0.028723634779453278], [Lrba, -0.028638655319809914], [Kmt2e, -0.028249293565750122], [Fam120a, -0.027979841455817223], [Polr2m, -0.027961425483226776], [Sun2, -0.027878060936927795], [Smad1, -0.027094585821032524], [Tmem123, -0.027037320658564568], [Prkar1a, -0.02697714790701866], [Ndfip1, -0.026941722258925438], [Dnaja1, -0.02690478041768074], [Psat1, -0.026514997705817223], [Ewsr1, -0.026310130953788757], [Hnrnph1, -0.026305943727493286], [Kctd12, -0.02629878744482994], [Prpf6, -0.02623290754854679], [Ctbp1, -0.026164360344409943], [Fth1, -0.026079533621668816], [Zfp36l1, -0.025776555761694908], [Tuba1c, -0.025710344314575195], [Uqcrh, -0.02525312453508377], ...]","[[Rpl34, 0.21652652323246002], [Gm13826, 0.2062893807888031], [Rpl18, 0.20176580548286438], [Rps19, 0.1820836365222931], [Gm15772, 0.17492258548736572], [Rps27rt, 0.16678665578365326], [Rpl23a, 0.15941938757896423], [Rps11, 0.1479083150625229], [Ubb, 0.13484805822372437], [Chd4, 0.1247732862830162], [Gm10845, 0.11338655650615692], [Rplp1, 0.11150107532739639], [Mef2d, 0.11008016765117645], [Rpl21, 0.10415194928646088], [Rpl41, 0.10233414173126221], [Tpt1, 0.10047775506973267], [Rsrp1, 0.0943563282489776


Drug_Repurposing_Hub.txt
Shape: (22621, 12)
Columns:
['!Source', 'The Drug Repurposing Hub, Broad Institute', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']


,!Source,"The Drug Repurposing Hub, Broad Institute",Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,!URL,http://www.broadinstitute.org/repurposing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,!File_date,8/18/2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,!Table_name,pert_ids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



LINCS_small_molecules.tsv
Shape: (34418, 8)
Columns:
['Unnamed: 0', 'pert_name', 'target', 'moa', 'canonical_smiles', 'inchi_key', 'compound_aliases', 'sig_count']


...

,rummageo_drug_name,pubchem_title,pubchem_cid,drh_moa,drh_target,drh_clinical_phase,drh_fda_approved,drh_disease_area,drh_indication,drh_inchi_key,drh_matched_on,moa_inherited_from
0,abemaciclib,Abemaciclib,46220502,CDK inhibitor,CDK4 | CDK6,Launched,yes,oncology,breast cancer,UZWDCWONPYILKI-UHFFFAOYSA-N,pubchem_cid,
1,acetaminophen,Acetaminophen,1983,cyclooxygenase inhibitor,CYP2E1 | FAAH | PTGS1 | PTGS2 | TRPV1,Launched,yes,endocrinology | neurology/psychiatry,fever | pain relief,"InChI=1S/C8H9NO2/c1-6(10)9-7-2-4-8(11)5-3-7/h2-5,11H,1H3,(H,9,10)",pubchem_cid,
2,acetylcysteine,N-Acetyl-L-Cysteine,12035,mucolytic agent,ACY1 | CHUK | GRIN1 | GRIN2A | GRIN2B | GRIN2D | GRIN3A | GSS | IKBKB | RELA | SLC7A11,Launched,yes,gastroenterology | gastroenterology,acetaminophen overdose | hepatic injury,PWKSKIMOESPYIA-BYPYZUCNSA-N,pubchem_cid,



sciplex-drug_metadata.parquet
Shape: (188, 12)
Columns:
['sciplex_drug_name', 'pubchem_title', 'pubchem_cid', 'drh_moa', 'drh_target', 'drh_clinical_phase', 'drh_fda_approved', 'drh_disease_area', 'drh_indication', 'drh_inchi_key', 'drh_matched_on', 'moa_inherited_from']


,sciplex_drug_name,pubchem_title,pubchem_cid,drh_moa,drh_target,drh_clinical_phase,drh_fda_approved,drh_disease_area,drh_indication,drh_inchi_key,drh_matched_on,moa_inherited_from
0,2-Methoxyestradiol,2-Methoxyestradiol,66414,hypoxia inducible factor inhibitor,CASP3 | COMT | CYP19A1 | CYP1A1 | CYP1B1 | HIF1A | TNFRSF10B | TUBB,Phase 2,no,,,CQOQDQWUFQDJMK-SSTWWWIQSA-N,pubchem_cid,
1,A-366,"5'-Methoxy-6'-(3-(pyrrolidin-1-yl)propoxy)spiro(cyclobutane-1,3'-indol)-2'-amine",76285486,histone lysine methyltransferase inhibitor,EHMT1 | EHMT2,Preclinical,no,,,BKCDJTRMYWSXMC-UHFFFAOYSA-N,pubchem_cid,
2,ABT-737,"4-(4-((4'-Chloro(1,1'-biphenyl)-2-yl)methyl)-1-piperazinyl)-N-((4-(((1R)-3-(dimethylamino)-1-((phenylthio)methyl)propyl)amino)-3-nitrophenyl)sulfonyl)benzamide",11228183,BCL inhibitor,BCL2 | BCL2L1 | BCL2L2,Phase 1/Phase 2,no,,,HPLNQCPCUACXLM-PGUFJCEWSA-N,pubchem_cid,



tahoe-100m-drug_metadata.parquet
Shape: (379, 18)
Columns:
['tahoe_drug_name', 'pubchem_title', 'pubchem_cid', 'smiles', 'tahoe_targets', 'tahoe_moa_broad', 'tahoe_moa_fine', 'tahoe_human_approved', 'tahoe_clinical_trials', 'drh_moa', 'drh_target', 'drh_clinical_phase', 'drh_fda_approved', 'drh_disease_area', 'drh_indication', 'drh_inchi_key', 'drh_matched_on', 'moa_inherited_from']


,tahoe_drug_name,pubchem_title,pubchem_cid,smiles,tahoe_targets,tahoe_moa_broad,tahoe_moa_fine,tahoe_human_approved,tahoe_clinical_trials,drh_moa,drh_target,drh_clinical_phase,drh_fda_approved,drh_disease_area,drh_indication,drh_inchi_key,drh_matched_on,moa_inherited_from
0,Talc,Talc,165411828,[OH-].[OH-].[O-][Si]12O[Si]3(O[Si](O1)(O[Si](O2)(O3)[O-])[O-])[O-].[Mg+2].[Mg+2].[Mg+2],NaN,unclear,unclear,yes,yes,,,,no,,,,name,
1,Bortezomib,Bortezomib,387447,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN=C2)(O)O,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,NFkB pathway inhibitor | proteasome inhibitor,CASP3 | CYP2C19 | NFKB1 | PSMA1 | PSMA2 | PSMA3 | PSMA4 | PSMA5 | PSMA6 | PSMA7 | PSMA8 | PSMB1 | PSMB10 | PSMB11 | PSMB2 | PSMB3 | PSMB4 | PSMB5 | PSMB6 | PSMB7 | PSMB8 | PSMB9 | PSMD1 | PSMD2 | RELA,Launched,yes,hematologic malignancy | hematologic malignancy,mantle cell lymphoma (MCL) | multiple myeloma,GXJABQQUPOEUTA-RDJZCZTQSA-N,pubchem_cid,
2,Ixazomib,Ixazomib,25183872,B(C(CC(C)C)NC(=O)CNC(=O)C1=C(C=CC(=C1)Cl)Cl)(O)O,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,proteasome inhibitor,,Launched,yes,hematologic malignancy,multiple myeloma,MXAYKZJJDUDWDS-LBPRGKRZSA-N,pubchem_cid,


In [11]:
from pathlib import Path
import pandas as pd
import json
import numpy as np

folder = Path("Step 1 Files")

def load_file(path):
    suffix = path.suffix.lower()

    if path.name == ".DS_Store":
        return None

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix in [".tsv", ".txt"]:
        return pd.read_csv(path, sep="\t", low_memory=False)

    if suffix == ".json":
        with open(path, "r") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.json_normalize(data)
        elif isinstance(data, dict):
            return pd.json_normalize(data)

    return None


def pick_col(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def get_series(df, col):
    if col is not None:
        return df[col]
    else:
        return pd.Series(pd.NA, index=df.index)


records = []

for path in sorted(folder.iterdir()):
    if path.name == ".DS_Store":
        continue

    df = load_file(path)
    if df is None:
        continue

    source = path.stem

    name_col = pick_col(df, [
        "drug_name", "compound_name", "pert_name", "pert_iname",
        "pubchem_title", "name", "drug", "lincs_drug_name",
        "nibr_drug_name", "tahoe_drug_name", "creeds_drug_name",
        "deepcover_drug_name", "rummageo_drug_name", "sciplex_drug_name",
        "cmap_drug_name"
    ])

    smiles_col = pick_col(df, [
        "smiles", "canonical_smiles"
    ])

    inchi_col = pick_col(df, [
        "inchi", "inchi_key", "inchikey", "drh_inchi_key"
    ])

    cid_col = pick_col(df, [
        "pubchem_cid", "pubchem_id", "cid", "pubchem"
    ])

    moa_col = pick_col(df, [
        "moa", "drh_moa", "moa-broad", "moa_broad", "moabox_moa",
        "moa-fine", "moa_fine"
    ])

    target_col = pick_col(df, [
        "target", "targets", "drh_target"
    ])

    pert_id_col = pick_col(df, [
        "pert_id", "pert_ids", "perturb_id", "lincs_pert_id"
    ])

    temp = pd.DataFrame({
        "source": pd.Series(source, index=df.index),
        "compound_name": get_series(df, name_col),
        "smiles": get_series(df, smiles_col),
        "inchi_key": get_series(df, inchi_col),
        "pubchem_cid": get_series(df, cid_col),
        "moa": get_series(df, moa_col),
        "target": get_series(df, target_col),
        "pert_id": get_series(df, pert_id_col),
    })

    records.append(temp)

registry_raw = pd.concat(records, ignore_index=True)

registry_raw.head()

,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id
0,CREEDS,Fluorouracil,C1=C(C(=O)NC(=O)N1)F,<NA>,3385.0,<NA>,<NA>,"[GSM26744, GSM26745]"
1,CREEDS,Resveratrol,Oc1ccc(cc1)/C=C/c1cc(O)cc(c1)O,<NA>,NaN,<NA>,<NA>,"[GSM801198, GSM801200, GSM801203, GSM801205, GSM801207, GSM801209, GSM801210, GSM801213, GSM801215, GSM801217]"
2,CREEDS,Citalopram,CN(C)CCCC1(C2=C(CO1)C=C(C=C2)C#N)C3=CC=C(C=C3)F,<NA>,2771.0,<NA>,<NA>,"[GSM162897, GSM162899, GSM162901]"
3,CREEDS,Fluorouracil,C1=C(C(=O)NC(=O)N1)F,<NA>,3385.0,<NA>,<NA>,"[GSM26742, GSM26743]"
4,CREEDS,Ethanol,CCO,<NA>,702.0,<NA>,<NA>,"[GSM1273500, GSM1273501, GSM1273502, GSM1273503]"


In [12]:
registry = registry_raw.copy()

for col in ["compound_name", "smiles", "inchi_key", "pubchem_cid", "moa", "target", "pert_id"]:
    registry[col] = registry[col].astype("string").str.strip()

registry = registry.replace({
    "": pd.NA,
    "nan": pd.NA,
    "None": pd.NA,
    "NA": pd.NA,
    "N/A": pd.NA,
    "-": pd.NA
})

print("Total rows:", len(registry))
print("Rows with SMILES:", registry["smiles"].notna().sum())
print("Rows with PubChem CID:", registry["pubchem_cid"].notna().sum())
print("Rows with InChIKey:", registry["inchi_key"].notna().sum())

registry.head()

Total rows: 87258
Rows with SMILES: 42893
Rows with PubChem CID: 26856
Rows with InChIKey: 40745


,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id
0,CREEDS,Fluorouracil,C1=C(C(=O)NC(=O)N1)F,<NA>,3385.0,<NA>,<NA>,"['GSM26744', 'GSM26745']"
1,CREEDS,Resveratrol,Oc1ccc(cc1)/C=C/c1cc(O)cc(c1)O,<NA>,<NA>,<NA>,<NA>,"['GSM801198', 'GSM801200', 'GSM801203', 'GSM801205', 'GSM801207', 'GSM801209', 'GSM801210', 'GSM801213', 'GSM801215', 'GSM801217']"
2,CREEDS,Citalopram,CN(C)CCCC1(C2=C(CO1)C=C(C=C2)C#N)C3=CC=C(C=C3)F,<NA>,2771.0,<NA>,<NA>,"['GSM162897', 'GSM162899', 'GSM162901']"
3,CREEDS,Fluorouracil,C1=C(C(=O)NC(=O)N1)F,<NA>,3385.0,<NA>,<NA>,"['GSM26742', 'GSM26743']"
4,CREEDS,Ethanol,CCO,<NA>,702.0,<NA>,<NA>,"['GSM1273500', 'GSM1273501', 'GSM1273502', 'GSM1273503']"


In [13]:
from rdkit import Chem
import pandas as pd

registry_with_smiles = registry[
    registry["smiles"].notna() &
    (registry["smiles"].astype(str).str.strip() != "")
].copy()

def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return pd.NA
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return pd.NA

registry_with_smiles["canonical_smiles"] = registry_with_smiles["smiles"].apply(canonicalize_smiles)

print("Rows before canonicalization:", len(registry_with_smiles))
print("Rows with valid canonical SMILES:", registry_with_smiles["canonical_smiles"].notna().sum())
print("Invalid SMILES:", registry_with_smiles["canonical_smiles"].isna().sum())
print("Unique canonical SMILES:", registry_with_smiles["canonical_smiles"].nunique())

[17:35:08] SMILES Parse Error: syntax error while parsing: restricted
[17:35:08] SMILES Parse Error: check for mistakes around position 1:
[17:35:08] restricted
[17:35:08] ^
[17:35:08] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'
[17:35:08] SMILES Parse Error: syntax error while parsing: restricted
[17:35:08] SMILES Parse Error: check for mistakes around position 1:
[17:35:08] restricted
[17:35:08] ^
[17:35:08] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'
[17:35:08] SMILES Parse Error: syntax error while parsing: restricted
[17:35:08] SMILES Parse Error: check for mistakes around position 1:
[17:35:08] restricted
[17:35:08] ^
[17:35:08] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'
[17:35:08] SMILES Parse Error: syntax error while parsing: restricted
[17:35:08] SMILES Parse Error: check for mistakes around position 1:
[17:35:08] restricted
[17:35:08] ^
[17:35:08] SMILES Parse Error: Fai

Rows before canonicalization: 42893
Rows with valid canonical SMILES: 42871
Invalid SMILES: 22
Unique canonical SMILES: 31711


In [14]:
valid_registry = registry_with_smiles[
    registry_with_smiles["canonical_smiles"].notna()
].copy()

def collapse_unique(series):
    vals = (
        series.dropna()
        .astype(str)
        .str.strip()
    )
    vals = vals[~vals.isin(["", "nan", "None", "<NA>", "NA", "N/A", "-"])]
    unique_vals = sorted(vals.unique())
    return "; ".join(unique_vals) if len(unique_vals) > 0 else pd.NA

compound_registry_clean = (
    valid_registry
    .groupby("canonical_smiles", as_index=False)
    .agg({
        "source": collapse_unique,
        "compound_name": collapse_unique,
        "smiles": collapse_unique,
        "inchi_key": collapse_unique,
        "pubchem_cid": collapse_unique,
        "moa": collapse_unique,
        "target": collapse_unique,
        "pert_id": collapse_unique,
    })
)

print("Clean unique compounds:", len(compound_registry_clean))
compound_registry_clean.head()

Clean unique compounds: 31711


,canonical_smiles,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id
0,Br.CN(C)CCCC1(c2ccc(F)cc2)OCc2cc(C#N)ccc21,Tahoe_100M; tahoe-100m-drug_metadata,Citalopram (hydrobromide); Citalopram Hydrobromide,CN(C)CCCC1(C2=C(CO1)C=C(C=C2)C#N)C3=CC=C(C=C3)F.Br,<NA>,77995; 77995.0,inhibitor/antagonist; selective serotonin reuptake inhibitor (SSRI),ADRA1A | CHRM1 | HRH1 | SLC6A2 | SLC6A3 | SLC6A4; SLC6A4,<NA>
1,BrC1C(Br)C(Br)C(Br)C(Br)C1Br,LINCS_small_molecules; lincs-l1000-cp-drug_metadata,"1,2,3,4,5,6-Hexabromocyclohexane; BRD-K06817181",BrC1C(Br)C(Br)C(Br)C(Br)C1Br,QFQZKISCBJKVHI-UHFFFAOYSA-N,74603,JAK inhibitor,JAK2,BRD-K06817181
2,Brc1c(Br)c(Br)c2[nH]nnc2c1Br,LINCS_small_molecules; lincs-l1000-cp-drug_metadata,"4,5,6,7-Tetrabromobenzotriazole; BRD-K97118047",Brc1c(Br)c(Br)c2[nH]nnc2c1Br,OMZYUVOATZSGJY-UHFFFAOYSA-N,1694,casein kinase inhibitor,AKT1 | CHEK1 | CSNK2A1 | CSNK2A2 | CSNK2B | GSK3B | LCK | MAP2K1 | MAPK1 | MAPK11 | MAPK12 | MAPK14 | MAPK8 | PRKCA | ROCK1 | RPS6KB1 | SGK1,BRD-K97118047
3,Brc1c(NC2=NCCN2)ccc2nccnc12,LINCS_small_molecules; Tahoe_100M; lincs-l1000-cp-drug_metadata; tahoe-100m-drug_metadata,Brimonidine; brimonidine,Brc1c(NC2=NCCN2)ccc2nccnc12; C1CN=C(N1)NC2=C(C3=NC=CN=C3C=C2)Br,XYLJNLCSTIOKRM-UHFFFAOYSA-N,2435; 2435.0,Adrenergic receptor agonist; activator/agonist; adrenergic receptor agonist,"ADRA2A | ADRA2B | ADRA2C | AOX1; ADRA2A, ADRA2B, ADRA2C; ADRA2B, ADRA2C, ADRA2A",BRD-K68264559
4,Brc1cc2c(cc1C1Nc3ccccc3C3C=CCC31)OCO2,LINCS_small_molecules; Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata; Novartis_metadata; lincs-l1000-cp-drug_metadata,"4-(6-bromo-1,3-benzodioxol-5-yl)-3a,4,5,9b-tetrahydro-3H-cyclopenta[c]quinoline; BRD-A83255679",Brc1cc2OCOc2cc1C3Nc4ccccc4C5C=CCC53; Brc1cc2c(cc1C1Nc3ccccc3C3C=CCC31)OCO2,YOLTZIVRJAPVPH-UHFFFAOYSA-N,3136844; 3136844.0,Estrogen receptor antagonist; GPER antagonist,GPER1; ['GPER1'],BRD-A83255679


In [15]:
compound_registry_clean.to_csv("step1_compound_registry_clean.csv", index=False)

print("Saved clean registry:")
print("step1_compound_registry_clean.csv")

Saved clean registry:
step1_compound_registry_clean.csv


In [16]:
from pathlib import Path

folder = Path("Step 1 Files")

list(folder.glob("*chembl*")), list(folder.glob("*CID*"))

([PosixPath('Step 1 Files/chembl_37_chemreps.txt.gz')],
 [PosixPath('Step 1 Files/CID-SMILES (1).gz')])

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("Step 1 Files")

chembl_path = list(folder.glob("chembl*_chemreps*.txt.gz"))[0]
pubchem_path = list(folder.glob("CID-SMILES*.gz"))[0]

print("ChEMBL:", chembl_path)
print("PubChem:", pubchem_path)

ChEMBL: Step 1 Files/chembl_37_chemreps.txt.gz
PubChem: Step 1 Files/CID-SMILES (1).gz


In [18]:
chembl = pd.read_csv(chembl_path, sep="\t", compression="gzip", low_memory=False)

print(chembl.shape)
print(chembl.columns.tolist())
chembl.head()

(2897819, 4)
['chembl_id', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key']


,chembl_id,canonical_smiles,standard_inchi,standard_inchi_key
0,CHEMBL153534,Cc1cc(-c2csc(N=C(N)N)n2)cn1C,"InChI=1S/C10H13N5S/c1-6-3-7(4-15(6)2)8-5-16-10(13-8)14-9(11)12/h3-5H,1-2H3,(H4,11,12,13,14)",MFRNFCWYPYSFQQ-UHFFFAOYSA-N
1,CHEMBL440060,CC[C@H](C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@@H](N)CCSC)[C@@H](C)O)C(=O)NCC(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](Cc1c[nH]cn1)C(=O)N[C@@H](CC(N)=O)C(=O)NCC(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCN=C(N)N)C(=O)NCC(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)NCC(=O)N1CCC[C@H]1C(=O)N1CCC[C@H]1C(=O)NCC(=O)N[C@@H](CO)C(=O)N[C@@H](CCCN=C(N)N)C(N)=O,"InChI=1S/C123H212N44O34S/c1-19-63(12)96(164-115(196)81(47-62(10)11)163-119(200)97(68(17)169)165-103(184)70(124)36-42-202-18)118(199)143-52-92(175)147-65(14)100(181)149-67(16)102(183)157-82(48-69-50-136-57-145-69)114(195)162-83(49-90(128)173)106(187)141-51-91(174)146-64(13)99(180)148-66(15)101(182)153-75(31-34-88(126)171)109(190)160-80(46-61(8)9)113(194)161-79(45-60(6)7)112(193)155-73(27-22-39-139-123(134)135)107(188)156-76(32-35-89(127)172)110(191)159-78(44-59(4)5)111(192)154-72(26-21-38-138-122(132)133)104(185)140-53-93(176)150-74(30-33-87(125)170)108(189)158-77(43-58(2)3)105(186)144-55-95(178)166-40-24-29-86(166)120(201)167-41-23-28-85(167)117(198)142-54-94(177)151-84(56-168)116(197)152-71(98(129)179)25-20-37-137-121(130)131/h50,57-68,70-86,96-97,168-169H,19-49,51-56,124H2,1-18H3,(H2,125,170)(H2,126,171)(H2,127,172)(H2,128,173)(H2,129,179)(H,136,145)(H,140,185)(H,141,187)(H,142,198)(H,143,199)(H,144,186)(H,146,174)(H,147,175)(H,148,180)(H,149,181)(H,150,176)(H,151,177)(H,152,197)(H,153,182)(H,154,192)(H,155,193)(H,156,188)(H,157,183)(H,158,189)(H,159,191)(H,160,190)(H,161,194)(H,162,195)(H,163,200)(H,164,196)(H,165,184)(H4,130,131,137)(H4,132,133,138)(H4,134,135,139)/t63-,64-,65-,66-,67-,68+,70-,71-,72-,73-,74-,75-,76-,77-,78-,79-,80-,81-,82-,83-,84-,85-,86-,96-,97-/m0/s1",RSEQNZQKBMRQNM-VRGFNVLHSA-N
2,CHEMBL440245,CCCC[C@@H]1NC(=O)[C@@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@H](N)Cc2ccccc2)C(C)C)CCC(=O)NCCCC[C@@H](C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](C)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](C)C(=O)N[C@@H](Cc2c[nH]cn2)C(=O)N[C@@H](CO)C(=O)N[C@@H](CC(N)=O)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCCN)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCC)C(=O)N[C@@H](CCC(=O)O)C(=O)N[C@H](C(=O)N[C@H](C(=O)C(N)=O)[C@@H](C)CC)[C@@H](C)CC)NC(=O)[C@H](C)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](C)NC1=O,"InChI=1S/C160H268N50O41/c1-23-27-41-95-134(228)182-88(20)130(224)187-99(45-36-62-177-158(168)169)135(229)183-87(19)129(223)186-97(44-33-35-61-176-121(216)57-51-105(143(237)189-95)198-150(244)112(69-83(13)14)206-156(250)124(84(15)16)208-145(239)106(52-58-122(217)218)196-140(234)101(47-38-64-179-160(172)173)192-149(243)110(67-81(9)10)203-151(245)111(68-82(11)12)204-152(246)114(72-93-75-175-78-181-93)200-133(227)94(162)70-91-39-30-29-31-40-91)138(232)195-104(50-56-119(165)214)144(238)201-108(65-79(5)6)147(241)185-89(21)131(225)188-103(49-55-118(164)213)142(236)194-102(48-54-117(163)212)136(230)184-90(22)132(226)199-113(71-92-74-174-77-180-92)153(247)207-116(76-211)155(249)205-115(73-120(166)215)154(248)193-100(46-37-63-178-159(170)171)139(233)190-98(43-32-34-60-161)141(235)202-109(66-80(7)8)148(242)191-96(42-28-24-2)137(231)197-107(53-59-123(219)220)146(240)210-126(86(18)26-4)157(251)209-125(85(17)25-3)127(221)128(167)222/h29-31,39-40,74-75,77-90,94-116,124-126,211H,23-28,32-38,41-73,76,161-162H2,1-22H3,(H2,163,212)(H2,164,213)(H2,165,214)(H2,166,215)(H2,167,222)(H,174,180)(H,175,181)(H,176,216)(H,182,228)(H,183,229)(H,184,230)(H,185,241)(H,186,223)(H,187,224)(H,188,225)(H,189,237)(H,190,233)(H,191,242)(H,192,243)(H,193,248)(H,194,236)(H,195,232)(H,196,234)(H,197,231)(H,198,244)(H,199,226)(H,200,

In [19]:
chembl_registry = pd.DataFrame({
    "source": "ChEMBL_37",
    "compound_name": chembl.get("chembl_id", pd.NA),
    "smiles": chembl.get("canonical_smiles", pd.NA),
    "inchi_key": chembl.get("standard_inchi_key", pd.NA),
    "pubchem_cid": pd.NA,
    "moa": pd.NA,
    "target": pd.NA,
    "pert_id": chembl.get("chembl_id", pd.NA),
})

chembl_registry.head()

,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id
0,ChEMBL_37,CHEMBL153534,Cc1cc(-c2csc(N=C(N)N)n2)cn1C,MFRNFCWYPYSFQQ-UHFFFAOYSA-N,<NA>,<NA>,<NA>,CHEMBL153534
1,ChEMBL_37,CHEMBL440060,CC[C@H](C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@@H](N)CCSC)[C@@H](C)O)C(=O)NCC(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](Cc1c[nH]cn1)C(=O)N[C@@H](CC(N)=O)C(=O)NCC(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCN=C(N)N)C(=O)NCC(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)NCC(=O)N1CCC[C@H]1C(=O)N1CCC[C@H]1C(=O)NCC(=O)N[C@@H](CO)C(=O)N[C@@H](CCCN=C(N)N)C(N)=O,RSEQNZQKBMRQNM-VRGFNVLHSA-N,<NA>,<NA>,<NA>,CHEMBL440060
2,ChEMBL_37,CHEMBL440245,CCCC[C@@H]1NC(=O)[C@@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@H](N)Cc2ccccc2)C(C)C)CCC(=O)NCCCC[C@@H](C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](C)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](CCC(N)=O)C(=O)N[C@@H](C)C(=O)N[C@@H](Cc2c[nH]cn2)C(=O)N[C@@H](CO)C(=O)N[C@@H](CC(N)=O)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCCN)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCC)C(=O)N[C@@H](CCC(=O)O)C(=O)N[C@H](C(=O)N[C@H](C(=O)C(N)=O)[C@@H](C)CC)[C@@H](C)CC)NC(=O)[C@H](C)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](C)NC1=O,FTKBTEIKPOYCEX-OZSLQWTKSA-N,<NA>,<NA>,<NA>,CHEMBL440245
3,ChEMBL_37,CHEMBL440249,CC(C)C[C@@H]1NC(=O)CNC(=O)[C@H](c2ccc(O)cc2)NC(=O)[C@@H]([C@@H](C)O)NC(=O)[C@H](c2ccc(O[C@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]3O[C@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]3O)cc2)NC(=O)[C@@H](CCCN)NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]([C@@H](C)O)NC(=O)[C@@H](c2ccc(O)cc2)NC(=O)[C@H](c2ccc(O)cc2)NC(=O)[C@@H](C(C)C)NC(=O)[C@@H](CCCN)NC(=O)[C@@H](c2ccc(O)cc2)NC(=O)[C@@H](CNC(=O)[C@H](CC(N)=O)NC(=O)Cc2cccc3ccccc23)[C@@H](C(N)=O)OC(=O)[C@H](c2ccc(O)c(Cl)c2)NC(=O)[C@@H](C)NC1=O,UYSXXKGACMHPIM-KFGDMSGDSA-N,<NA>,<NA>,<NA>,CHEMBL440249
4,ChEMBL_37,CHEMBL405398,Brc1cccc(Nc2ncnc3ccncc23)c1NCCN1CCOCC1,VDSXZXJEWIWBCG-UHFFFAOYSA-N,<NA>,<NA>,<NA>,CHEMBL405398


In [20]:
N_PUBCHEM = 1_000_000  # increase to 2_000_000 or 5_000_000 later if your computer is fine

pubchem = pd.read_csv(
    pubchem_path,
    sep=r"\s+",
    compression="gzip",
    header=None,
    names=["pubchem_cid", "smiles"],
    nrows=N_PUBCHEM
)

print(pubchem.shape)
pubchem.head()

(1000000, 2)


,pubchem_cid,smiles
0,1,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C
1,2,CC(=O)OC(CC(=O)O)C[N+](C)(C)C
2,3,C1=CC(C(C(=C1)C(=O)O)O)O
3,4,CC(CN)O
4,5,C(C(=O)COP(=O)(O)O)N


In [21]:
pubchem_registry = pd.DataFrame({
    "source": "PubChem_CID_SMILES",
    "compound_name": pd.NA,
    "smiles": pubchem["smiles"],
    "inchi_key": pd.NA,
    "pubchem_cid": pubchem["pubchem_cid"],
    "moa": pd.NA,
    "target": pd.NA,
    "pert_id": pd.NA,
})

pubchem_registry.head()

,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id
0,PubChem_CID_SMILES,<NA>,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,<NA>,1,<NA>,<NA>,<NA>
1,PubChem_CID_SMILES,<NA>,CC(=O)OC(CC(=O)O)C[N+](C)(C)C,<NA>,2,<NA>,<NA>,<NA>
2,PubChem_CID_SMILES,<NA>,C1=CC(C(C(=C1)C(=O)O)O)O,<NA>,3,<NA>,<NA>,<NA>
3,PubChem_CID_SMILES,<NA>,CC(CN)O,<NA>,4,<NA>,<NA>,<NA>
4,PubChem_CID_SMILES,<NA>,C(C(=O)COP(=O)(O)O)N,<NA>,5,<NA>,<NA>,<NA>


In [22]:
existing_registry = pd.read_csv("step1_compound_registry_clean.csv")

# Make sure existing registry has the same columns
needed_cols = ["source", "compound_name", "smiles", "inchi_key", "pubchem_cid", "moa", "target", "pert_id"]

for col in needed_cols:
    if col not in existing_registry.columns:
        existing_registry[col] = pd.NA

existing_registry = existing_registry[needed_cols]
chembl_registry = chembl_registry[needed_cols]
pubchem_registry = pubchem_registry[needed_cols]

expanded_raw = pd.concat(
    [existing_registry, chembl_registry, pubchem_registry],
    ignore_index=True
)

print("Existing registry:", len(existing_registry))
print("ChEMBL added:", len(chembl_registry))
print("PubChem added:", len(pubchem_registry))
print("Expanded raw total:", len(expanded_raw))
expanded_raw.head()

Existing registry: 31711
ChEMBL added: 2897819
PubChem added: 1000000
Expanded raw total: 3929530


,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id
0,Tahoe_100M; tahoe-100m-drug_metadata,Citalopram (hydrobromide); Citalopram Hydrobromide,CN(C)CCCC1(C2=C(CO1)C=C(C=C2)C#N)C3=CC=C(C=C3)F.Br,NaN,77995; 77995.0,inhibitor/antagonist; selective serotonin reuptake inhibitor (SSRI),ADRA1A | CHRM1 | HRH1 | SLC6A2 | SLC6A3 | SLC6A4; SLC6A4,NaN
1,LINCS_small_molecules; lincs-l1000-cp-drug_metadata,"1,2,3,4,5,6-Hexabromocyclohexane; BRD-K06817181",BrC1C(Br)C(Br)C(Br)C(Br)C1Br,QFQZKISCBJKVHI-UHFFFAOYSA-N,74603,JAK inhibitor,JAK2,BRD-K06817181
2,LINCS_small_molecules; lincs-l1000-cp-drug_metadata,"4,5,6,7-Tetrabromobenzotriazole; BRD-K97118047",Brc1c(Br)c(Br)c2[nH]nnc2c1Br,OMZYUVOATZSGJY-UHFFFAOYSA-N,1694,casein kinase inhibitor,AKT1 | CHEK1 | CSNK2A1 | CSNK2A2 | CSNK2B | GSK3B | LCK | MAP2K1 | MAPK1 | MAPK11 | MAPK12 | MAPK14 | MAPK8 | PRKCA | ROCK1 | RPS6KB1 | SGK1,BRD-K97118047
3,LINCS_small_molecules; Tahoe_100M; lincs-l1000-cp-drug_metadata; tahoe-100m-drug_metadata,Brimonidine; brimonidine,Brc1c(NC2=NCCN2)ccc2nccnc12; C1CN=C(N1)NC2=C(C3=NC=CN=C3C=C2)Br,XYLJNLCSTIOKRM-UHFFFAOYSA-N,2435; 2435.0,Adrenergic receptor agonist; activator/agonist; adrenergic receptor agonist,"ADRA2A | ADRA2B | ADRA2C | AOX1; ADRA2A, ADRA2B, ADRA2C; ADRA2B, ADRA2C, ADRA2A",BRD-K68264559
4,LINCS_small_molecules; Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata; Novartis_metadata; lincs-l1000-cp-drug_metadata,"4-(6-bromo-1,3-benzodioxol-5-yl)-3a,4,5,9b-tetrahydro-3H-cyclopenta[c]quinoline; BRD-A83255679",Brc1cc2OCOc2cc1C3Nc4ccccc4C5C=CCC53; Brc1cc2c(cc1C1Nc3ccccc3C3C=CCC31)OCO2,YOLTZIVRJAPVPH-UHFFFAOYSA-N,3136844; 3136844.0,Estrogen receptor antagonist; GPER antagonist,GPER1; ['GPER1'],BRD-A83255679


In [23]:
from rdkit import Chem

def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return pd.NA
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return pd.NA

expanded_raw = expanded_raw[
    expanded_raw["smiles"].notna() &
    (expanded_raw["smiles"].astype(str).str.strip() != "")
].copy()

expanded_raw["canonical_smiles"] = expanded_raw["smiles"].apply(canonicalize_smiles)

print("Rows before RDKit filtering:", len(expanded_raw))
print("Valid canonical SMILES:", expanded_raw["canonical_smiles"].notna().sum())
print("Invalid SMILES:", expanded_raw["canonical_smiles"].isna().sum())
print("Unique canonical SMILES:", expanded_raw["canonical_smiles"].nunique())

[19:48:11] SMILES Parse Error: syntax error while parsing: Brc1c(NC2=NCCN2)ccc2nccnc12;
[19:48:11] SMILES Parse Error: check for mistakes around position 28:
[19:48:11] C2=NCCN2)ccc2nccnc12;
[19:48:11] ~~~~~~~~~~~~~~~~~~~~^
[19:48:11] SMILES Parse Error: Failed parsing SMILES 'Brc1c(NC2=NCCN2)ccc2nccnc12;' for input: 'Brc1c(NC2=NCCN2)ccc2nccnc12;'
[19:48:11] SMILES Parse Error: syntax error while parsing: Brc1cc2OCOc2cc1C3Nc4ccccc4C5C=CCC53;
[19:48:11] SMILES Parse Error: check for mistakes around position 36:
[19:48:11] C3Nc4ccccc4C5C=CCC53;
[19:48:11] ~~~~~~~~~~~~~~~~~~~~^
[19:48:11] SMILES Parse Error: Failed parsing SMILES 'Brc1cc2OCOc2cc1C3Nc4ccccc4C5C=CCC53;' for input: 'Brc1cc2OCOc2cc1C3Nc4ccccc4C5C=CCC53;'
[19:48:11] SMILES Parse Error: syntax error while parsing: C#CCN(C)[C@H](C)Cc1ccccc1;
[19:48:11] SMILES Parse Error: check for mistakes around position 26:
[19:48:11] (C)[C@H](C)Cc1ccccc1;
[19:48:11] ~~~~~~~~~~~~~~~~~~~~^
[19:48:11] SMILES Parse Error: Failed parsing SMILES '

Rows before RDKit filtering: 3929530
Valid canonical SMILES: 3928538
Invalid SMILES: 992
Unique canonical SMILES: 3765890


In [24]:
valid_expanded = expanded_raw[expanded_raw["canonical_smiles"].notna()].copy()

def collapse_unique(series):
    vals = series.dropna().astype(str).str.strip()
    vals = vals[~vals.isin(["", "nan", "None", "<NA>", "NA", "N/A", "-"])]
    unique_vals = sorted(vals.unique())
    return "; ".join(unique_vals) if len(unique_vals) > 0 else pd.NA

expanded_registry_clean = (
    valid_expanded
    .groupby("canonical_smiles", as_index=False)
    .agg({
        "source": collapse_unique,
        "compound_name": collapse_unique,
        "smiles": collapse_unique,
        "inchi_key": collapse_unique,
        "pubchem_cid": collapse_unique,
        "moa": collapse_unique,
        "target": collapse_unique,
        "pert_id": collapse_unique,
    })
)

print("Expanded clean unique compounds:", len(expanded_registry_clean))
expanded_registry_clean.head()

KeyboardInterrupt: 

In [25]:
valid_expanded = expanded_raw[expanded_raw["canonical_smiles"].notna()].copy()

# Prefer rows from your curated perturbation datasets over giant PubChem rows
def get_priority(source):
    source = str(source).lower()
    if "pubchem" in source:
        return 3
    elif "chembl" in source:
        return 2
    else:
        return 1

valid_expanded["source_priority"] = valid_expanded["source"].apply(get_priority)

# Prefer rows that have more useful metadata
valid_expanded["has_name"] = valid_expanded["compound_name"].notna().astype(int)
valid_expanded["has_moa"] = valid_expanded["moa"].notna().astype(int)
valid_expanded["has_target"] = valid_expanded["target"].notna().astype(int)
valid_expanded["has_inchi"] = valid_expanded["inchi_key"].notna().astype(int)
valid_expanded["has_pubchem"] = valid_expanded["pubchem_cid"].notna().astype(int)

valid_expanded_sorted = valid_expanded.sort_values(
    by=[
        "source_priority",
        "has_moa",
        "has_target",
        "has_name",
        "has_inchi",
        "has_pubchem"
    ],
    ascending=[True, False, False, False, False, False]
)

expanded_registry_clean = (
    valid_expanded_sorted
    .drop_duplicates(subset="canonical_smiles", keep="first")
    .drop(columns=[
        "source_priority",
        "has_name",
        "has_moa",
        "has_target",
        "has_inchi",
        "has_pubchem"
    ])
    .reset_index(drop=True)
)

print("Expanded clean unique compounds:", len(expanded_registry_clean))
expanded_registry_clean.head()

Expanded clean unique compounds: 3765890


,source,compound_name,smiles,inchi_key,pubchem_cid,moa,target,pert_id,canonical_smiles
0,LINCS_small_molecules; lincs-l1000-cp-drug_metadata,"1,2,3,4,5,6-Hexabromocyclohexane; BRD-K06817181",BrC1C(Br)C(Br)C(Br)C(Br)C1Br,QFQZKISCBJKVHI-UHFFFAOYSA-N,74603,JAK inhibitor,JAK2,BRD-K06817181,BrC1C(Br)C(Br)C(Br)C(Br)C1Br
1,LINCS_small_molecules; lincs-l1000-cp-drug_metadata,"4,5,6,7-Tetrabromobenzotriazole; BRD-K97118047",Brc1c(Br)c(Br)c2[nH]nnc2c1Br,OMZYUVOATZSGJY-UHFFFAOYSA-N,1694,casein kinase inhibitor,AKT1 | CHEK1 | CSNK2A1 | CSNK2A2 | CSNK2B | GSK3B | LCK | MAP2K1 | MAPK1 | MAPK11 | MAPK12 | MAPK14 | MAPK8 | PRKCA | ROCK1 | RPS6KB1 | SGK1,BRD-K97118047,Brc1c(Br)c(Br)c2[nH]nnc2c1Br
2,Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata; Novartis_metadata,"7-Bromo-1,2,3,4-tetrahydroisoquinoline; 7-bromo-1,2,3,4-tetrahydroisoquinoline",Brc1ccc2c(c1)CNCC2,OYODEQFZAJVROF-UHFFFAOYSA-N,10729255; 10729255.0,Phenylethanolamine N-methyltransferase inhibitor,['PNMT'],NaN,Brc1ccc2c(c1)CNCC2
3,Novartis:DRUG-seq U2OS MoABox Dataset-drug-seq-drug_metadata; Novartis_metadata,"4-(3-bromophenyl)-1H-1,2,3-triazole; 4-(3-bromophenyl)-2H-triazole",Brc1cccc(-c2c[nH]nn2)c1,BKJXSQXZIOTRTP-UHFFFAOYSA-N,6400901; 6400901.0,Methionine Aminopeptidase-2 (MetAP2) Inhibitors;Angiogenesis Inhibitors,['METAP2'],NaN,Brc1cccc(-c2c[nH]nn2)c1
4,Tahoe_100M; tahoe-100m-drug_metadata,Quinestrol,CC12CCC3C(C1CCC2(C#C)O)CCC4=C3C=CC(=C4)OC5CCCC5,PWZUUYSISTUNDW-VAFBSOEGSA-N,9046; 9046.0,estrogen receptor agonist; unclear,ESR1 | ESR2,NaN,C#CC1(O)CCC2C3CCc4cc(OC5CCCC5)ccc4C3CCC21C


In [26]:
expanded_registry_clean.to_csv("step1_compound_registry_expanded_clean_fast.csv", index=False)

print("Saved fast clean registry:")
print("step1_compound_registry_expanded_clean_fast.csv")

Saved fast clean registry:
step1_compound_registry_expanded_clean_fast.csv


In [27]:
import gzip
from pathlib import Path

folder = Path("Step 1 Files")
pubchem_path = list(folder.glob("CID-SMILES*.gz"))[0]

total_pubchem_rows = 0
with gzip.open(pubchem_path, "rt") as f:
    for _ in f:
        total_pubchem_rows += 1

print("Total PubChem rows in file:", total_pubchem_rows)
print("PubChem rows used:", N_PUBCHEM)
print("PubChem rows left out:", total_pubchem_rows - N_PUBCHEM)

Total PubChem rows in file: 123994425
PubChem rows used: 1000000
PubChem rows left out: 122994425


In [28]:
from pathlib import Path

summary = """
Step 1 Compound Registry Expansion

Goal:
Build compound registry for compound → compound autoencoder.

Sources used:
- PerturbSeqr drug metadata
- LINCS small molecules
- Drug Repurposing Hub
- CREEDS chemical perturbations
- Novartis DRUG-seq metadata
- Tahoe 100M metadata
- L1000-CP drug metadata
- CMAP/microarray drug metadata
- SciPlex drug metadata
- DeepCoverMOA drug metadata
- RummaGEO chemical drug metadata
- ChEMBL 37 chemreps
- PubChem CID-SMILES subset

Key outputs:
- step1_compound_registry_clean.csv
- step1_compound_registry_expanded_clean_fast.csv

Current expanded clean registry:
- 3,765,890 unique valid canonical SMILES

Important note:
Only 1,000,000 rows from PubChem CID-SMILES were used for computational feasibility.
Full PubChem CID-SMILES file contained 123,994,425 rows.
"""

Path("step1_registry_summary.txt").write_text(summary)

print("Saved step1_registry_summary.txt")

Saved step1_registry_summary.txt
